> ### Data Cleaning

In [0]:
df = spark.table("workspace.default.messy_customers_large")
display(df)

CustomerID,Name,Age,City,Salary,JoinDate
1,Kiran,44,bangalore,NULL,2023-04-01
2,Meena,24,BANGALORE,27891,2023-09-04
3,John,null,chennai,not_available,2023-01-04
4,Meena,null,Pune,40297,2023-05-18
5,Asha,39,Delhi,NULL,2023-08-28
6,Sneha,null,delhi,not_available,2023-01-10
7,Manoj,abc,Delhi,37696,2023-09-09
8,Rahul,21,Chennai,NULL,2023-04-27
9,Priya,abc,bangalore,NULL,2023-09-16
10,Ravi,abc,Bangalore,37334,2023-04-22


In [0]:
df.filter(df.Age.isNull()).show()

+----------+------+----+---------+-------------+----------+
|CustomerID|  Name| Age|     City|       Salary|  JoinDate|
+----------+------+----+---------+-------------+----------+
|         3|  John|NULL|  chennai|not_available|2023-01-04|
|         4| Meena|NULL|     Pune|        40297|2023-05-18|
|         6| Sneha|NULL|    delhi|not_available|2023-01-10|
|        13| Suraj|NULL|    Delhi|         NULL|2023-06-28|
|        16| Rahul|NULL|Hyderabad|not_available|2023-04-18|
|        21| Kumar|NULL|      blr|         NULL|2023-04-20|
|        22| Divya|NULL|  chennai|        26094|2023-07-03|
|        23|  Sana|NULL|    Delhi|         NULL|2023-10-25|
|        26| Divya|NULL|    Delhi|        37909|2023-12-18|
|        30| Rahul|NULL|      blr|         NULL|2023-02-24|
|        43|  John|NULL|Hyderabad|         NULL|2023-10-06|
|        46| Manoj|NULL|   Mumbai|         NULL|2023-02-13|
|        47|Vikram|NULL|Bangalore|         NULL|2023-07-04|
|        49| Sneha|NULL|   Mumbai|      

In [0]:
from pyspark.sql.functions import expr

df = df.withColumn("Age",expr("try_cast(Age as Double)"))

In [0]:
from pyspark.sql.functions import *
avg_age = df.select(round(avg("Age"),2)).collect()[0][0]
print(avg_age)

32.64


In [0]:
from pyspark.sql.functions import *
# age_int = df.replace({None:avg_age},subset= ["Age"]) --- why cant we use cuz replace is for string or particular value not for null 
# age_int.filter(df.Age == "abc").show()
df = df.fillna(avg_age, subset=["Age"])
df = df.withColumn("Age", round("Age",0))
df.select("Age").show()

+----+
| Age|
+----+
|44.0|
|24.0|
|33.0|
|33.0|
|39.0|
|33.0|
|33.0|
|21.0|
|33.0|
|33.0|
|33.0|
|31.0|
|33.0|
|21.0|
|33.0|
|33.0|
|20.0|
|33.0|
|33.0|
|33.0|
+----+
only showing top 20 rows


In [0]:
df.show()

+----------+------+----+---------+-------------+----------+
|CustomerID|  Name| Age|     City|       Salary|  JoinDate|
+----------+------+----+---------+-------------+----------+
|         1| Kiran|44.0|bangalore|         NULL|2023-04-01|
|         2| Meena|24.0|BANGALORE|        27891|2023-09-04|
|         3|  John|33.0|  chennai|not_available|2023-01-04|
|         4| Meena|33.0|     Pune|        40297|2023-05-18|
|         5|  Asha|39.0|    Delhi|         NULL|2023-08-28|
|         6| Sneha|33.0|    delhi|not_available|2023-01-10|
|         7| Manoj|33.0|    Delhi|        37696|2023-09-09|
|         8| Rahul|21.0|  Chennai|         NULL|2023-04-27|
|         9| Priya|33.0|bangalore|         NULL|2023-09-16|
|        10|  Ravi|33.0|Bangalore|        37334|2023-04-22|
|        11|  Sana|33.0|bangalore|        63409|2023-01-16|
|        12| Priya|31.0|Bangalore|        63617|2023-02-18|
|        13| Suraj|33.0|    Delhi|         NULL|2023-06-28|
|        14|Vikram|21.0|  Chennai|      

In [0]:
df = df.replace({"BANGALORE":"Bangalore", "bangalore":"Bangalore", "delhi":"Delhi","HYDERABAD":"Hyderabad", "CHENNAI":"Chennai","blr":"Bangalore","Bengaluru":"Bangalore","chennai":"Chennai"}, subset=["City"])
df.show()

+----------+------+----+---------+--------+----------+
|CustomerID|  Name| Age|     City|  Salary|  JoinDate|
+----------+------+----+---------+--------+----------+
|         1| Kiran|44.0|Bangalore|51865.66|2023-04-01|
|         2| Meena|24.0|Bangalore| 27891.0|2023-09-04|
|         3|  John|33.0|  Chennai|51865.66|2023-01-04|
|         4| Meena|33.0|     Pune| 40297.0|2023-05-18|
|         5|  Asha|39.0|    Delhi|51865.66|2023-08-28|
|         6| Sneha|33.0|    Delhi|51865.66|2023-01-10|
|         7| Manoj|33.0|    Delhi| 37696.0|2023-09-09|
|         8| Rahul|21.0|  Chennai|51865.66|2023-04-27|
|         9| Priya|33.0|Bangalore|51865.66|2023-09-16|
|        10|  Ravi|33.0|Bangalore| 37334.0|2023-04-22|
|        11|  Sana|33.0|Bangalore| 63409.0|2023-01-16|
|        12| Priya|31.0|Bangalore| 63617.0|2023-02-18|
|        13| Suraj|33.0|    Delhi|51865.66|2023-06-28|
|        14|Vikram|21.0|  Chennai| 43571.0|2023-10-15|
|        15| Kiran|33.0|Hyderabad|51865.66|2023-05-18|
|        1

In [0]:
from pyspark.sql.functions import *
df = df.withColumn("Salary", expr("try_cast(Salary as Double)"))
# df = df.withColumn(
#     "Salary",
#     col("Salary").cast("double")
# )


In [0]:
df.show()

+----------+------+----+---------+-------+----------+
|CustomerID|  Name| Age|     City| Salary|  JoinDate|
+----------+------+----+---------+-------+----------+
|         1| Kiran|44.0|Bengaluru|   NULL|2023-04-01|
|         2| Meena|24.0|Bengaluru|27891.0|2023-09-04|
|         3|  John|33.0|  chennai|   NULL|2023-01-04|
|         4| Meena|33.0|     Pune|40297.0|2023-05-18|
|         5|  Asha|39.0|    Delhi|   NULL|2023-08-28|
|         6| Sneha|33.0|    Delhi|   NULL|2023-01-10|
|         7| Manoj|33.0|    Delhi|37696.0|2023-09-09|
|         8| Rahul|21.0|  Chennai|   NULL|2023-04-27|
|         9| Priya|33.0|Bengaluru|   NULL|2023-09-16|
|        10|  Ravi|33.0|Bangalore|37334.0|2023-04-22|
|        11|  Sana|33.0|Bengaluru|63409.0|2023-01-16|
|        12| Priya|31.0|Bangalore|63617.0|2023-02-18|
|        13| Suraj|33.0|    Delhi|   NULL|2023-06-28|
|        14|Vikram|21.0|  Chennai|43571.0|2023-10-15|
|        15| Kiran|33.0|Hyderabad|   NULL|2023-05-18|
|        16| Rahul|33.0|Hyde

In [0]:
avg_salary = df.select(avg("Salary")).collect()[0][0]
print(avg_salary)

51865.66129032258


In [0]:
df = df.fillna(avg_salary,"Salary")
df = df.withColumn("Salary", round("Salary",2))
df.select("Salary").show()


+--------+
|  Salary|
+--------+
|51865.66|
| 27891.0|
|51865.66|
| 40297.0|
|51865.66|
|51865.66|
| 37696.0|
|51865.66|
|51865.66|
| 37334.0|
| 63409.0|
| 63617.0|
|51865.66|
| 43571.0|
|51865.66|
|51865.66|
|51865.66|
|51865.66|
|51865.66|
| 48277.0|
+--------+
only showing top 20 rows


In [0]:
employee_Count_City = df.groupBy("City").count().show()

+---------+-----+
|     City|count|
+---------+-----+
|Bangalore|   72|
|  Chennai|   29|
|     Pune|   14|
|    Delhi|   33|
|Hyderabad|   36|
|   Mumbai|   16|
+---------+-----+



In [0]:
df.display()

CustomerID,Name,Age,City,Salary,JoinDate
1,Kiran,44.0,Bangalore,51865.66,2023-04-01
2,Meena,24.0,Bangalore,27891.0,2023-09-04
3,John,33.0,Chennai,51865.66,2023-01-04
4,Meena,33.0,Pune,40297.0,2023-05-18
5,Asha,39.0,Delhi,51865.66,2023-08-28
6,Sneha,33.0,Delhi,51865.66,2023-01-10
7,Manoj,33.0,Delhi,37696.0,2023-09-09
8,Rahul,21.0,Chennai,51865.66,2023-04-27
9,Priya,33.0,Bangalore,51865.66,2023-09-16
10,Ravi,33.0,Bangalore,37334.0,2023-04-22
